# 🦌 Introducing IPyElk 👋

This is a brief introduction to the JupyterLab widget wrapper for displaying diagrams
using the Eclipse Layout Kernel - [elkjs](https://github.com/kieler/elkjs) and
[sprotty](https://github.com/eclipse/sprotty).

In [ ]:
from importlib.util import find_spec

if find_spec("ipyelk") is None and __name__ == "__main__":
    import sys
    from subprocess import check_call

    check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
    ])

In [ ]:
import json
import pathlib

import ipywidgets
import traitlets

import ipyelk

# ElkDiagram

The `ElkDiagram` is a low-level widget that accepts and validates a dictionary against
the [Elk JSON Schema](../elk/schema/elkschema.json).

> _Note: cell below will update with the input below: try **Create New View For Output**
> from the output context menu to see the changes._

In [ ]:
def a_simple_elk_json_example(elk_json_path="simple.json", **kwargs):
    if "layout" not in kwargs:
        kwargs["layout"] = dict(height="100%", min_height="400px")
    elk_json = json.loads(pathlib.Path(elk_json_path).read_text(encoding="utf-8"))
    loader = ipyelk.ElkJSONLoader().clear_defaults()
    return ipyelk.Diagram(source=loader.load(elk_json), **kwargs)


if __name__ == "__main__":
    simple = a_simple_elk_json_example()
    display(simple)

## 🎛 Interaction

Elk diagrams are highly interactive. Some of these interactions are event-driven, while
others are available as _traits_ which can be modified directly, or linked between
diagrams.

### 🎯 Centering

By default, `center` will move the center of the diagram to the center of the view.

In [ ]:
if __name__ == "__main__":
    simple.view.center()

Optionally, specific node/edges can be provided, animation can be disabled, and zoom
preserved.

In [ ]:
if __name__ == "__main__":
    simple.view.center(["svg1", "kernel"], animate=False, retain_zoom=True)

### 📏 Fitting

By default, `fit` will fill the view with the diagram.

In [ ]:
if __name__ == "__main__":
    simple.view.fit()

Optionally, provide animation can be disabled, and zoom limited, and padding added.

In [ ]:
if __name__ == "__main__":
    simple.view.fit(["kernel"], animate=False, max_zoom=999, padding=5)

### 🏷 Selecting

The currently-selected nodes and edges are available

In [ ]:
if __name__ == "__main__":
    selected = ipywidgets.Text(description="selected", disabled=True)
    traitlets.dlink((simple.view.selection, "ids"), (selected, "value"), " and ".join)
    display(selected)

Clicking a single element will mark it as _selected_.

In [ ]:
if __name__ == "__main__":
    simple.view.selection.ids = ["kernel"]

Holding down <kbd>Ctrl</kbd> while clicking will select multiple nodes.

In [ ]:
if __name__ == "__main__":
    simple.view.selection.ids = ["kernel", "e1"]

Clicking outside of any node will clear the selection.

In [ ]:
if __name__ == "__main__":
    simple.view.selection.ids = []

### 🛸 Hovering

The element under the pointer is also traited: `hovered_id` is its id, or `None`
when the pointer is over no element

In [ ]:
if __name__ == "__main__":
    hovered = ipywidgets.Text(description="hovered", disabled=True)
    traitlets.dlink(
        (simple.view.hover, "hovered_id"),
        (hovered, "value"),
        lambda v: v or "",  # None (pointer left) shows as an empty label
    )
    display(hovered)

In [ ]:
if __name__ == "__main__":
    simple.view.hover.hovered_id = "user"  # highlight from the kernel; None clears it

### 🎥 Viewport

The browser reports the camera of the most recently reporting view as one snapshot:
`view_id`, `origin`, `zoom`, `canvas_size` and `viewed_ids` (the elements whose bounds
touch the visible rectangle). The traits are read-only in the kernel and `None` until
a view reports.

In [ ]:
if __name__ == "__main__":
    viewed = ipywidgets.Text(description="viewed", disabled=True)
    traitlets.dlink(
        (simple.view.viewport, "viewed_ids"),
        (viewed, "value"),
        lambda ids: "" if ids is None else " and ".join(ids),
    )
    display(viewed)

To move the camera, send a command: `None` keeps the current origin or zoom, and
`view_id` addresses one view (every view by default).

In [ ]:
if __name__ == "__main__":
    simple.view.set_viewport(origin=(0, 0), zoom=1.5, animate=False)

### 🎨 Painting

`painter` adds CSS classes to rendered elements in every view without touching the
model: a re-layout keeps them, exported SVG shows them, and the ELK JSON never changes.
`unpaint` removes classes (or, with none given, the ids) and `clear` removes everything.

In [ ]:
if __name__ == "__main__":
    simple.style = {" .highlight .elknode": {"fill": "var(--jp-warn-color3)"}}
    simple.view.painter.paint(["kernel", "user"], "highlight")
    print(simple.view.painter.styles)

In [ ]:
if __name__ == "__main__":
    simple.view.painter.clear()

## 🦌 Learn More 📖

See the [other examples](./_index.ipynb).